# Part-1 Step 1 — Data Exploration & Schema Discovery

**Goal**: Load the Amazon Fashion dataset (30k products), profile every attribute, document data quality findings, and identify the usable features for the similarity search engine.

This notebook covers:
1. Loading & initial inspection
2. Full column schema
3. PLAN feature ↔ dataset column mapping
4. Missing value analysis
5. Numeric distributions
6. Categorical cardinality
7. Image URL availability
8. Text field statistics
9. Edge cases & data quality notes
10. Recommended feature set for the similarity engine

In [4]:
# --- Imports & Constants ---
import json
import math
from pathlib import Path

import pandas as pd

# Sentinel value used in the raw dataset to represent missing weight
WEIGHT_SENTINEL = 999_999_999

# Path to the raw ldjson dataset (relative to this notebook)
DATA_PATH = Path("../data/marketing_sample_for_amazon_com-amazon_fashion_products__20200201_20200430__30k_data.ldjson")

# Column groupings — these drive all the analysis sections below
NUMERIC_COLS = ["sales_price", "rating", "weight", "discount_percentage"]
CATEGORICAL_COLS = ["brand", "colour", "delivery_type", "amazon_prime__y_or_n", "best_seller_tag__y_or_n"]
TEXT_COLS = ["product_name", "meta_keywords", "other_items_customers_buy"]
IMAGE_COLS = ["image_urls__small", "medium", "large"]

## §1 — Load Dataset & Overview

Load the `.ldjson` file line by line, coerce numeric columns, and replace the `999999999` weight sentinel with `NaN`.

In [ ]:
# Load each JSON line into a list of dicts, then build a DataFrame
records = []
with DATA_PATH.open() as fh:
    for line in fh:
        line = line.strip()
        if line:
            records.append(json.loads(line))

df = pd.read_json(DATA_PATH, lines=True)

# --- Clean up numeric columns ---
# Weight: replace the sentinel value 999999999 with NaN (it means "not available")
df["weight"] = pd.to_numeric(df["weight"], errors="coerce")
df.loc[df["weight"] == WEIGHT_SENTINEL, "weight"] = float("nan")

# Coerce other numeric-looking columns from strings to proper floats
for col in ["sales_price", "rating", "discount_percentage",
            "no__of_reviews", "no__of_offers", "no__of_sellers", "left_in_stock"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

print(f"Loaded {len(df):,} rows × {len(df.columns)} columns")

Loaded 30,000 rows × 33 columns


In [6]:
# --- Dataset overview ---
n_rows, n_cols = df.shape
n_dup = df["uniq_id"].duplicated().sum()
# Rows where more than half of the columns are null → candidates to drop (PLAN threshold)
n_null_rows = (df.isnull().sum(axis=1) / n_cols > 0.5).sum()
mem_mb = df.memory_usage(deep=True).sum() / 1_048_576
date_range = f"{df['crawl_timestamp'].min()[:10]} → {df['crawl_timestamp'].max()[:10]}"

print(f"Total rows:                {n_rows:,}")
print(f"Total columns:             {n_cols}")
print(f"Duplicate uniq_id:         {n_dup}")
print(f"Rows >50% missing (drop):  {n_null_rows}")
print(f"Memory usage:              {mem_mb:.1f} MB")
print(f"Crawl date range:          {date_range}")

Total rows:                30,000
Total columns:             33
Duplicate uniq_id:         0
Rows >50% missing (drop):  284
Memory usage:              96.1 MB
Crawl date range:          2020-02-06 → 2020-02-07


## §2 — Full Column Schema

Show every column with its dtype, non-null count, and missing percentage.
This tells us which columns are dense (always present) vs sparse (mostly missing).

In [7]:
# Build a summary table: column name, dtype, non-null count, missing %
n = len(df)
schema_rows = []
for col in df.columns:
    non_null = int(df[col].notna().sum())
    miss_pct = 100 * (n - non_null) / n
    sample = str(df[col].dropna().iloc[0])[:60] if non_null > 0 else ""
    schema_rows.append({
        "column": col,
        "dtype": str(df[col].dtype),
        "non_null": non_null,
        "missing_%": round(miss_pct, 1),
        "sample": sample,
    })

schema_df = pd.DataFrame(schema_rows)
schema_df.index += 1  # 1-based numbering
schema_df

,column,dtype,non_null,missing_%,sample
1,uniq_id,str,30000,0.0,26d41bdc1495de290bc8e6062d927729
2,crawl_timestamp,str,30000,0.0,2020-02-07 05:11:36 +0000
3,asin,str,30000,0.0,B07STS2W9T
4,product_url,str,30000,0.0,https://www.amazon.in/Facon-Kalamkari-Handbloc...
5,product_name,str,30000,0.0,LA' Facon Cotton Kalamkari Handblock Saree Blo...
6,image_urls__small,str,29998,0.0,https://images-na.ssl-images-amazon.com/images...
7,medium,str,29998,0.0,https://images-na.ssl-images-amazon.com/images...
8,large,str,28841,3.9,https://images-na.ssl-images-amazon.com/images...
9,browsenode,str,29480,1.7,1968255031
10,brand,str,21857,27.1,LA' Facon


## §3 — Feature ↔ Dataset Column Mapping

The feature specification references `color`, `price`, and `description` — but the dataset
uses different names. This mapping catches the mismatches early so we code against the
**actual** column names.

| Spec Feature | Dataset Column | Note |
|---|---|---|
| `color` | `colour` | British spelling |
| `price` | `sales_price` | No separate `price` column exists |
| `description` | `meta_keywords` | No `description` column; `meta_keywords` is the best text proxy |
| `image_urls` | `image_urls__small` / `medium` / `large` | Pipe-separated URLs |

In [ ]:
# Map: what the spec calls the feature → what the dataset actually has
PLAN_FEATURE_MAP = {
    "uniq_id":     "uniq_id",
    "product_name":"product_name",
    "brand":       "brand",
    "color":       "colour",           # spec says "color", dataset says "colour"
    "price":       "sales_price",      # spec says "price", only "sales_price" exists
    "sales_price": "sales_price",
    "weight":      "weight",
    "rating":      "rating",
    "description": "meta_keywords",    # no "description" col; meta_keywords is the proxy
    "image_urls":  "image_urls__small",
}

# Show coverage for each mapped feature
map_rows = []
for plan_name, actual_col in PLAN_FEATURE_MAP.items():
    present = actual_col in df.columns
    non_null = int(df[actual_col].notna().sum()) if present else 0
    miss_pct = round(100 * (n - non_null) / n, 1) if present else 100.0
    map_rows.append({
        "plan_feature": plan_name,
        "dataset_column": actual_col,
        "present": "✓" if present else "✗",
        "non_null": non_null,
        "missing_%": miss_pct,
    })

pd.DataFrame(map_rows)

,plan_feature,dataset_column,present,non_null,missing_%
0,uniq_id,uniq_id,✓,30000,0.0
1,product_name,product_name,✓,30000,0.0
2,brand,brand,✓,21857,27.1
3,color,colour,✓,6029,79.9
4,price,sales_price,✓,27110,9.6
5,sales_price,sales_price,✓,27110,9.6
6,weight,weight,✓,0,100.0
7,rating,rating,✓,30000,0.0
8,description,meta_keywords,✓,30000,0.0
9,image_urls,image_urls__small,✓,29998,0.0


## §4 — Missing Value Analysis

Sorted from most-missing to least-missing. The bar gives a visual sense of sparsity.
Rows with >50% missing columns are drop candidates (the preprocessing pipeline will remove them).

In [9]:
# Count nulls per column, sort descending, and add a text bar for quick visual
missing = (
    df.isnull()
    .sum()
    .rename("missing_count")
    .reset_index()
    .rename(columns={"index": "column"})
)
missing["missing_%"] = round(100 * missing["missing_count"] / n, 1)
missing = missing.sort_values("missing_%", ascending=False).reset_index(drop=True)

# Add a simple text-bar column (each █ = ~3%)
missing["bar"] = missing["missing_%"].apply(
    lambda p: "█" * max(0, round(p * 35 / 100)) + "░" * (35 - max(0, round(p * 35 / 100)))
)
missing

,column,missing_count,missing_%,bar
0,weight,30000,100.0,███████████████████████████████████
1,formats___editions,29998,100.0,███████████████████████████████████
2,name_of_author_for_books,29999,100.0,███████████████████████████████████
3,discount_percentage,30000,100.0,███████████████████████████████████
4,no__of_offers,28980,96.6,██████████████████████████████████░
5,no__of_sellers,28980,96.6,██████████████████████████████████░
6,technical_details__k_v_pairs,28846,96.2,██████████████████████████████████░
7,left_in_stock,26943,89.8,███████████████████████████████░░░░
8,no__of_reviews,26548,88.5,███████████████████████████████░░░░
9,colour,23971,79.9,████████████████████████████░░░░░░░


## §5 — Numeric Feature Distributions

Statistics for `sales_price`, `rating`, `weight`, and `discount_percentage`.

**Key things to look for:**
- Outliers (e.g. extreme max values) → justifies StandardScaler over MinMaxScaler
- Zero values that may represent missing data
- Skew that affects imputation strategy (median is more robust than mean)

In [10]:
# Compute descriptive statistics for each numeric column
numeric_cols_present = [c for c in NUMERIC_COLS if c in df.columns]

stats_rows = []
for col in numeric_cols_present:
    s = df[col].dropna()
    if s.empty:
        # Column exists but is entirely NaN (e.g. weight after sentinel cleanup)
        stats_rows.append({"feature": col, "count": 0, "note": "all NaN after cleanup"})
        continue
    desc = s.describe(percentiles=[0.25, 0.5, 0.75, 0.95])
    zeros = int((s == 0).sum())
    stats_rows.append({
        "feature": col,
        "count": int(desc["count"]),
        "min": round(float(s.min()), 2),
        "p25": round(float(desc["25%"]), 2),
        "median": round(float(desc["50%"]), 2),
        "mean": round(float(desc["mean"]), 2),
        "p75": round(float(desc["75%"]), 2),
        "p95": round(float(desc["95%"]), 2),
        "max": round(float(s.max()), 2),
        "std": round(float(desc["std"]), 2),
        "zeros": zeros,
    })

stats_df = pd.DataFrame(stats_rows)
stats_df

,feature,count,min,p25,median,mean,p75,p95,max,std,zeros,note
0,sales_price,27110,39.0,379.0,590.0,862.17,899.0,2877.1,9988.0,964.22,0.0,NaN
1,rating,30000,1.0,3.5,4.0,4.04,4.9,5.0,5.0,0.84,0.0,NaN
2,weight,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,all NaN after cleanup
3,discount_percentage,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,all NaN after cleanup


In [11]:
# --- Data quality warnings for numeric columns ---
print("⚠  weight: ALL values were the sentinel 999999999 → replaced with NaN.")
print("   This means weight is entirely missing for all 30k products.\n")

sp_zeros = int((df["sales_price"] == 0).sum()) if "sales_price" in df.columns else 0
if sp_zeros:
    print(f"⚠  sales_price: {sp_zeros:,} zero-price rows (likely missing or free items)")

rating_out = df["rating"].dropna()
out_of_range = int(((rating_out < 1) | (rating_out > 5)).sum())
if out_of_range:
    print(f"⚠  rating: {out_of_range:,} values outside expected [1, 5] range")
else:
    print("✓  rating: all values within [1, 5]")

⚠  weight: ALL values were the sentinel 999999999 → replaced with NaN.
   This means weight is entirely missing for all 30k products.

✓  rating: all values within [1, 5]


## §6 — Categorical Feature Cardinality

For each categorical column, show the top values and overall cardinality.
This informs the encoding strategy:
- **Low cardinality** (e.g. `delivery_type` with 2 values) → one-hot encoding
- **High cardinality** (e.g. `brand` with 6k+ values) → label encoding or frequency encoding

In [12]:
# For each categorical column, show cardinality and top-15 values
for col in CATEGORICAL_COLS:
    if col not in df.columns:
        continue
    vc = df[col].value_counts(dropna=True)
    total_non_null = int(vc.sum())
    cardinality = len(vc)

    print(f"\n{'='*60}")
    print(f"  {col}  |  {total_non_null:,} non-null  |  cardinality = {cardinality:,}")
    print(f"{'='*60}")

    # Show top 15 values with counts and percentages
    top_n = min(15, len(vc))
    for val, cnt in vc.head(top_n).items():
        pct_nn = 100 * cnt / total_non_null
        pct_all = 100 * cnt / n
        print(f"  {str(val):<40s}  {cnt:>6,}  ({pct_nn:5.1f}% of non-null, {pct_all:5.1f}% of total)")
    if cardinality > top_n:
        print(f"  ... {cardinality - top_n:,} more values")


  brand  |  21,857 non-null  |  cardinality = 6,458
  Max                                          504  (  2.3% of non-null,   1.7% of total)
  Generic                                      245  (  1.1% of non-null,   0.8% of total)
  BIBA                                         205  (  0.9% of non-null,   0.7% of total)
  Mothercare                                   156  (  0.7% of non-null,   0.5% of total)
  Campus Sutra                                 150  (  0.7% of non-null,   0.5% of total)
  Soch                                         149  (  0.7% of non-null,   0.5% of total)
  nauti nati                                   132  (  0.6% of non-null,   0.4% of total)
  Ada                                          127  (  0.6% of non-null,   0.4% of total)
  GRITSTONES                                   110  (  0.5% of non-null,   0.4% of total)
  PrintOctopus                                 102  (  0.5% of non-null,   0.3% of total)
  Allen Solly Junior                           

## §7 — Image URL Availability

Each image column holds pipe-separated (`|`) URLs. We need to know:
- How many products have at least one image URL (coverage)
- Average number of URLs per product (tells us which resolution tier to use)
- We'll use the **first URL** from `image_urls__small` or `medium` for feature extraction

In [13]:
# Check image URL availability for each size tier
img_rows = []
for col in IMAGE_COLS:
    if col not in df.columns:
        img_rows.append({"column": col, "has_url": 0, "coverage_%": 0})
        continue
    # A row "has a URL" if it's not null and not an empty string
    has_url = df[col].notna() & (df[col].str.strip() != "")
    coverage = int(has_url.sum())
    # Count how many pipe-separated URLs each product has
    url_counts = df.loc[has_url, col].str.split("|").str.len()
    img_rows.append({
        "column": col,
        "has_url": coverage,
        "coverage_%": round(100 * coverage / n, 1),
        "avg_urls_per_product": round(float(url_counts.mean()), 1),
        "max_urls_per_product": int(url_counts.max()),
    })

pd.DataFrame(img_rows)

,column,has_url,coverage_%,avg_urls_per_product,max_urls_per_product
0,image_urls__small,29998,100.0,4.2,13
1,medium,29998,100.0,4.2,13
2,large,28841,96.1,4.0,13


## §8 — Text Field Statistics

Text fields are the highest-value features for similarity search (they become
384-dimensional embeddings via sentence-transformers).

Key questions:
- Are there empty strings hiding behind non-null status?
- How long are the texts? Very short texts produce noisy embeddings.

In [14]:
# Measure text field length, emptiness, and show a sample snippet
text_rows = []
for col in TEXT_COLS:
    if col not in df.columns:
        continue
    s = df[col].fillna("").astype(str)
    non_null = int(df[col].notna().sum())
    empty = int((s.str.strip() == "").sum())
    lengths = s[s.str.strip() != ""].str.len()
    sample = str(df[col].dropna().iloc[0])[:80] if non_null > 0 else ""
    text_rows.append({
        "column": col,
        "non_null": non_null,
        "empty_strings": empty,
        "avg_length_chars": round(float(lengths.mean()), 0) if not lengths.empty else None,
        "min_length": int(lengths.min()) if not lengths.empty else None,
        "max_length": int(lengths.max()) if not lengths.empty else None,
        "sample": sample,
    })

pd.DataFrame(text_rows)

,column,non_null,empty_strings,avg_length_chars,min_length,max_length,sample
0,product_name,30000,0,58.0,4,500,LA' Facon Cotton Kalamkari Handblock Saree Blo...
1,meta_keywords,30000,0,78.0,9,742,LA' Facon Cotton Kalamkari Handblock Saree Blo...
2,other_items_customers_buy,24363,5637,500.0,6,3507,Cotton Kalamkari Handblock Saree Blouse/Kurti ...


## §9 — Edge Cases & Data Quality Notes

A catalogue of all data quality issues found above. These directly inform:
- **Test fixtures** in Step 2 (edge-case records with missing fields)
- **Preprocessing rules** in Step 3 (sentinel handling, imputation strategy)
- **Feature extraction fallbacks** in Steps 5-7 (zero vectors for missing data)

In [ ]:
# Collect all data quality issues into a structured list
issues = []

# 1. Weight sentinel
issues.append(("weight", "ALL 30k values were sentinel 999999999 → replaced with NaN. Effectively 100% missing."))

# 2. Zero prices
sp_zeros = int((df["sales_price"] == 0).sum()) if "sales_price" in df.columns else 0
if sp_zeros:
    issues.append(("sales_price", f"{sp_zeros:,} zero-price rows → likely missing or free items"))

# 3. Ratings outside [1, 5]
out_of_range = int(((df["rating"].dropna() < 1) | (df["rating"].dropna() > 5)).sum())
if out_of_range:
    issues.append(("rating", f"{out_of_range:,} values outside [1, 5]"))

# 4. Duplicate IDs
dup = int(df["uniq_id"].duplicated().sum())
if dup:
    issues.append(("uniq_id", f"{dup:,} duplicate IDs"))
else:
    issues.append(("uniq_id", "0 duplicates — safe to use as primary key ✓"))

# 5. Rows with >50% missing columns (drop threshold)
drop_candidates = int((df.isnull().sum(axis=1) / len(df.columns) > 0.5).sum())
issues.append(("(rows)", f"{drop_candidates:,} rows have >50% missing columns → drop candidates"))

# 6. Column naming mismatches between spec and dataset
issues.append(("colour", "Spec uses 'color'; dataset column is 'colour' (British spelling)"))
issues.append(("price", "Spec uses 'price'; only 'sales_price' exists in dataset"))
issues.append(("description", "No 'description' column; best proxy is 'meta_keywords'"))

# 7. Sparse columns not worth using as features
if "no__of_reviews" in df.columns:
    cov = round(100 * df["no__of_reviews"].notna().sum() / n, 1)
    issues.append(("no__of_reviews", f"only {cov}% coverage — too sparse for a feature"))

if "colour" in df.columns:
    cov = round(100 * df["colour"].notna().sum() / n, 1)
    issues.append(("colour (coverage)", f"only {cov}% coverage with {df['colour'].nunique():,} unique values"))

# Display as a DataFrame for readability
pd.DataFrame(issues, columns=["field", "issue"])

,field,issue
0,weight,ALL 30k values were sentinel 999999999 → repla...
1,uniq_id,0 duplicates — safe to use as primary key ✓
2,(rows),284 rows have >50% missing columns → drop cand...
3,colour,PLAN_1 references 'color'; dataset column is '...
4,price,PLAN_1 references 'price'; only 'sales_price' ...
5,description,PLAN_1 references 'description'; best proxy is...
6,no__of_reviews,only 11.5% coverage — too sparse for a feature
7,colour (coverage),"only 20.1% coverage with 4,757 unique values"


## §10 — Recommended Feature Set for Similarity Engine

Based on the profiling above, here is the recommended feature set for the multimodal
similarity engine:

| Column | Modality | Coverage | Encoding | Note |
|---|---|---|---|---|
| `product_name` | text | 100% | all-MiniLM-L6-v2 (384-d) | Highest value; always present |
| `meta_keywords` | text (proxy for description) | 100% | concat with product_name | No `description` col |
| `image_urls__small` | image | ~100% | EfficientNet-B0 (1280-d) | Use first URL; fallback = zero vector |
| `sales_price` | numeric | ~90% | StandardScaler | ~10% missing → impute with median |
| `rating` | numeric | 100% | StandardScaler | Always present |
| `weight` | numeric | 0% | StandardScaler | All sentinel → impute median (likely uninformative) |
| `brand` | categorical | ~73% | LabelEncoder | ~27% missing → fill with "unknown" |
| `colour` | categorical | ~20% | LabelEncoder | ~80% missing — minimal signal |

### Design Decisions
- **Similarity weights**: text=0.4, image=0.3, structured=0.3
- **Fallback** (no image): text=0.6, structured=0.4
- **Tie-breaking**: secondary sort by `sales_price` ascending (cheaper first)
- **Drop rule**: rows with >50% missing columns are dropped before feature extraction
- **StandardScaler > MinMaxScaler**: robust to price outliers (max price >> median)

In [16]:
# Summary table of recommended features with coverage percentages
recs = [
    ("uniq_id",           "identity",         "100.0%", "—",                        "Primary key — not a feature"),
    ("product_name",      "text",             f"{100*df['product_name'].notna().sum()/n:.1f}%",
                                              "all-MiniLM-L6-v2 (384-d)",           "Always present; highest value"),
    ("meta_keywords",     "text (proxy)",     f"{100*df['meta_keywords'].notna().sum()/n:.1f}%",
                                              "concat with product_name",           "No 'description' col; use this"),
    ("image_urls__small", "image",            f"{100*(df['image_urls__small'].notna()&(df['image_urls__small'].str.strip()!='')).sum()/n:.1f}%",
                                              "EfficientNet-B0 (1280-d)",           "First URL only; fallback=zero vec"),
    ("sales_price",       "numeric",          f"{100*df['sales_price'].notna().sum()/n:.1f}%",
                                              "StandardScaler",                     "~10% missing → impute median"),
    ("rating",            "numeric",          f"{100*df['rating'].notna().sum()/n:.1f}%",
                                              "StandardScaler",                     "Always present"),
    ("weight",            "numeric",          f"{100*df['weight'].notna().sum()/n:.1f}%",
                                              "StandardScaler",                     "Sentinel→NaN; 100% missing"),
    ("brand",             "categorical",      f"{100*df['brand'].notna().sum()/n:.1f}%",
                                              "LabelEncoder",                       "~27% missing → fill 'unknown'"),
    ("colour",            "categorical",      f"{100*df['colour'].notna().sum()/n:.1f}%",
                                              "LabelEncoder",                       "~80% missing — low signal"),
]

pd.DataFrame(recs, columns=["column", "modality", "coverage", "encoding", "note"])

,column,modality,coverage,encoding,note
0,uniq_id,identity,100.0%,—,Primary key — not a feature
1,product_name,text,100.0%,all-MiniLM-L6-v2 (384-d),Always present; highest value
2,meta_keywords,text (proxy),100.0%,concat with product_name,No 'description' col; use this
3,image_urls__small,image,100.0%,EfficientNet-B0 (1280-d),First URL only; fallback=zero vec
4,sales_price,numeric,90.4%,StandardScaler,~10% missing → impute median
5,rating,numeric,100.0%,StandardScaler,Always present
6,weight,numeric,0.0%,StandardScaler,Sentinel→NaN; 100% missing
7,brand,categorical,72.9%,LabelEncoder,~27% missing → fill 'unknown'
8,colour,categorical,20.1%,LabelEncoder,~80% missing — low signal
